# 1. Import Libraries

In [1]:
import sys
from pathlib import Path

# Thêm thư mục gốc (project root) vào Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

import numpy as np

from utils.features import build_features

import optuna

# 2. Load Dataset

In [2]:
df_train, df_val = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = None,
                                pls_fit_years = [1]
                                   )

In [3]:
features = [
    'Year', 'Month', 'Day', 'Hour',
    'Combined_Temp',
    'Combined_GHI_1', 'Combined_GHI_2',
    'sin_day_k1', 'cos_day_k1'
]
target = 'Load'

# 3. Split Data

In [4]:
# Year 1 → TRAIN
train_mask = df_train['Year'] == 1

# Year 2 → VALIDATION
val_mask = df_val['Year'] == 2

# Year 3 → TEST
# Testing.xlsx is assumed to contain Year 3

X_train = df_train.loc[train_mask, features]
y_train = df_train.loc[train_mask, target]

X_val = df_val.loc[val_mask, features]
y_val = df_val.loc[val_mask, target]

print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")

Dataset sizes:
Train (Year 1): 8,784
Validation (Year 2): 8,760


# 4. Train on year 1, evaluate on year 2

In [5]:
tscv = TimeSeriesSplit(n_splits=4)

In [6]:
def objective(trial):

    params = {
        "objective": "reg:squarederror",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1500
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            10,
            log=True
        ),

        "random_state": 42,
        "n_jobs": -1
    }

    fold_rmse = []

    # ============================================
    # Time-series CV trên YEAR 1
    # ============================================

    for train_idx, val_idx in tscv.split(X_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)

        model.fit(
            X_fold_train,
            y_fold_train
        )

        y_pred = model.predict(
            X_fold_val
        )

        rmse = root_mean_squared_error(
            y_fold_val,
            y_pred
        )

        fold_rmse.append(rmse)

    # Optuna minimize mean CV RMSE
    return np.mean(fold_rmse)

In [7]:
study = optuna.create_study(
    direction="minimize",
    study_name="xgb_year1_timeseries_cv"
)

study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)

[I 2026-10-02 18:33:04,301] A new study created in memory with name: xgb_year1_timeseries_cv


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-10-02 18:33:05,542] Trial 0 finished with value: 284.00377655029297 and parameters: {'n_estimators': 685, 'learning_rate': 0.044736908001172616, 'max_depth': 4, 'min_child_weight': 6, 'subsample': 0.7888584027887461, 'colsample_bytree': 0.930961321832862, 'gamma': 3.542986991070409, 'reg_alpha': 8.937226797318606e-07, 'reg_lambda': 0.1035546113101104}. Best is trial 0 with value: 284.00377655029297.
[I 2026-10-02 18:33:11,916] Trial 1 finished with value: 310.5249710083008 and parameters: {'n_estimators': 1266, 'learning_rate': 0.016043210664298455, 'max_depth': 9, 'min_child_weight': 1, 'subsample': 0.779804911302939, 'colsample_bytree': 0.8270278165523471, 'gamma': 4.155792418863769, 'reg_alpha': 1.007153749335402e-08, 'reg_lambda': 0.007305652877433276}. Best is trial 0 with value: 284.00377655029297.
[I 2026-10-02 18:33:13,392] Trial 2 finished with value: 308.4929428100586 and parameters: {'n_estimators': 832, 'learning_rate': 0.03172378269423598, 'max_depth': 6, 'min_chil

In [8]:
print("Best CV RMSE:")
print(study.best_value)

print("\nBest parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best CV RMSE:
274.45149993896484

Best parameters:
n_estimators: 736
learning_rate: 0.024661591078667625
max_depth: 3
min_child_weight: 5
subsample: 0.6446702352361038
colsample_bytree: 0.913082298530404
gamma: 3.30729385200618
reg_alpha: 7.120808185588972e-06
reg_lambda: 0.0013814972364974353


In [9]:
best_params = study.best_params

seeds = [0, 1, 2, 3, 42]

results = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(X_val)

    results.append({
        "Seed": seed,
        "RMSE": root_mean_squared_error(y_val, y_pred),
        "MSE": mean_squared_error(y_val, y_pred),
        "MAE": mean_absolute_error(y_val, y_pred),
        "MAPE": mean_absolute_percentage_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred)
    })

stability_df = pd.DataFrame(results)

stability_summary = stability_df.drop(
    columns="Seed"
).agg(["mean", "std", "min", "max"]).T

print("=== Results by seed ===")
display(stability_df)

print("\n=== Stability summary ===")
display(stability_summary)

=== Results by seed ===


,Seed,RMSE,MSE,MAE,MAPE,R2
0,0,192.156494,36924.117188,141.414642,0.067464,0.776500
1,1,191.290802,36592.171875,140.713531,0.067055,0.778509
2,2,191.591248,36707.207031,140.878052,0.067172,0.777813
3,3,191.130035,36530.691406,140.598541,0.067050,0.778881
4,42,191.407745,36636.925781,140.793381,0.067152,0.778238



=== Stability summary ===


,mean,std,min,max
RMSE,191.515265,0.396010,191.130035,192.156494
MSE,36678.222656,151.805676,36530.691406,36924.117188
MAE,140.879630,0.316338,140.598541,141.414642
MAPE,0.067179,0.000169,0.067050,0.067464
R2,0.777988,0.000919,0.776500,0.778881


In [10]:
stability_df.to_csv("../report/seed_daily_first_harmonic_val_report.csv", index = False)
stability_summary.to_csv("../report/seed_summary_daily_first_harmonic_val_report.csv", index = False)

# 4. Final Training on year 1 and year 2

In [11]:
df_train, df_test = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = pd.read_excel("../Dataset/testing.xlsx")
                                   )

In [12]:
X_train = df_train[features]
y_train = df_train[target]

X_test = df_test[features]

In [13]:
print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")
print(f"Test (Year 3): {len(X_test):,}")

Dataset sizes:
Train (Year 1): 17,544
Validation (Year 2): 8,760
Test (Year 3): 8,760


In [16]:
seeds = [0, 1, 2, 3, 42]

test_predictions = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(X_test)

    test_predictions.append(pred)

y_test_pred = np.mean(
    test_predictions,
    axis=0
)

In [17]:
pd.DataFrame(y_test_pred).to_csv("../report/daily_first_harmonic_test_prediction.csv", index = False)